# Week 2: Agribusiness Data Cleaning & Preprocessing Framework
## Notebook 01: Interactive Cleaning, Harmonization & Quality Scorecard Demonstration

**Project Title**: Agribusiness Data Cleaning & Preprocessing Framework  
**Role**: Data Scientist Intern  
**Current Phase**: Week 2 – Data Cleaning and Preprocessing Strategy  

> **IMPORTANT DISCLAIMER**  
> All datasets utilized in this notebook are **HYPOTHETICAL EXAMPLES** synthesized strictly for demonstrating data-quality defect detection, transformation logic, and validation scoring. In accordance with internship requirements, no real-world statistics or findings are fabricated.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure repository src modules are accessible
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data_cleaning import AgribusinessCleaner
from src.validation import DataQualityValidator
from src.utils import setup_logger

logger = setup_logger()
print("Environment initialized successfully from:", repo_root)

### 1. Generation of Hypothetical Raw Agribusiness Dataset
We create a representative **HYPOTHETICAL** raw agricultural dataset containing common data quality defects:
- **Typographical & Whitespace Variations**: `" rice "`, `"RICE"`, `"Paddy"`
- **Legacy Geographic Names**: `"Orissa"`, `"Uttaranchal"`
- **Missing Values**: Uninformative nulls in `Production` and `Area`
- **Exact Duplicate Records**: Identical observations on composite primary keys
- **Invalid Values**: Mathematically impossible negative values (`Area = -50.0`, `Modal_Price = -450`)
- **Unit Inconsistencies / Extreme Outliers**: A cereal yield calculation exceeding biological ceilings ($>25,000$ kg/ha due to unit mismatch)

In [ ]:
# [HYPOTHETICAL EXAMPLE DATASET]
raw_hypothetical_data = pd.DataFrame({
    "State_Name": ["Punjab", "Punjab", "Orissa", "Uttaranchal", "Haryana", "Haryana", "Punjab", "Punjab"],
    "District_Name": ["Ludhiana", "Ludhiana", "Cuttack", "Dehradun", "Karnal", "Karnal", "Amritsar", "Ludhiana"],
    "Crop_Year": [2021, 2021, 2021, 2021, 2021, 2021, 2021, 2021],
    "Season": ["Rabi ", "Rabi ", "Kharif", "Kharif", "Rabi", "Rabi", "Rabi", "Rabi "],
    "Crop": ["Wheat", "Wheat", " rice ", "PADDY", "Gram", "Chickpea", "Wheat", "Wheat"],
    "Area": [1500.0, 1500.0, 800.0, -50.0, 400.0, 400.0, 1200.0, 1500.0],  # -50.0 is invalid
    "Production": [7200.0, 7200.0, 2400.0, 110.0, np.nan, 850.0, 36000.0, 7200.0],  # 36000 is extreme outlier
    "Modal_Price": [2015.0, 2015.0, 1860.0, 1940.0, 5100.0, 5100.0, 1980.0, 2015.0],
    "Arrival_Date": ["2021-04-10", "2021-04-10", "2021-11-15", "15/11/2021", "2021-04-05", "2021-04-05", "2021-04-12", "2021-04-10"]
})

print("Raw Hypothetical Dataset (Shape:", raw_hypothetical_data.shape, "):")
display(raw_hypothetical_data)

### 2. Pre-Cleaning Data Quality Audit (Baseline Scorecard)
Before performing transformations, we audit the hypothetical dataset using `DataQualityValidator` across completeness, uniqueness, validity, and mathematical consistency.

In [ ]:
baseline_validator = DataQualityValidator(raw_hypothetical_data, dataset_name="Raw_Hypothetical_Dataset")
baseline_scorecard = baseline_validator.generate_scorecard(
    critical_columns=["Area", "Production", "Modal_Price"],
    composite_key=["State_Name", "District_Name", "Crop_Year", "Season", "Crop", "Arrival_Date"]
)

print("=== Baseline Data Quality Scorecard (Pre-Cleaning) ===")
display(baseline_scorecard)

### 3. Execution of Agribusiness Data Preprocessing Pipeline
We now pass the raw dataset through the standardized stages of `AgribusinessCleaner`:
1. Column standardisation to `snake_case`
2. Categorical entity standardisation & crop taxonomy mapping
3. Composite-key deduplication
4. Invalid and uninformative missing value filtering
5. Unit normalization and yield calculation ($kg/ha$ and $t/ha$)
6. Multi-tiered outlier detection (biological boundaries & statistical IQR fences)
7. Feature engineering (calendar attributes, moving averages)

In [ ]:
cleaner = AgribusinessCleaner(raw_hypothetical_data)

cleaned_df = (
    cleaner.standardize_column_names()
           .standardize_categorical_entities()
           .deduplicate_composite_key()
           .handle_invalid_and_missing_values()
           .compute_yield_and_normalize_units()
           .detect_and_flag_outliers(iqr_multiplier=3.0)
           .engineer_analytical_features()
           .get_cleaned_data()
)

print("Cleaned & Preprocessed Dataset (Shape:", cleaned_df.shape, "):")
display(cleaned_df)

### 4. Review of Data Cleaning Audit Trail
The cleaner automatically tracks every modification, record count delta, and impacted volume to ensure full reproducibility.

In [ ]:
audit_trail = cleaner.get_audit_trail()
print("=== Transformation Audit Trail ===")
display(audit_trail)

### 5. Post-Cleaning Data Quality Scorecard
Auditing the cleaned dataset to confirm that completeness, uniqueness, and consistency have achieved 100% compliance.

In [ ]:
post_validator = DataQualityValidator(cleaned_df, dataset_name="Cleaned_Hypothetical_Dataset")
post_scorecard = post_validator.generate_scorecard(
    critical_columns=["area", "production", "modal_price", "yield_kg_per_ha"],
    composite_key=["state_name", "district_name", "crop_year", "season", "crop", "date"]
)

print("=== Post-Cleaning Data Quality Scorecard ===")
display(post_scorecard)

### 6. Summary of Key Learnings for Week 3 EDA
- **Deduplication**: Composite keys successfully caught duplicate entries (Ludhiana Wheat records).
- **Taxonomy Normalization**: Inconsistent casing (`" rice "`, `"PADDY"`) and state naming (`"Orissa"` $\rightarrow$ `"Odisha"`) were cleanly resolved.
- **Yield Verification**: An extreme yield of 30,000 kg/ha in Amritsar was flagged with `is_outlier = True` (`Biological yield violation for Cereals`), enabling researchers to isolate or audit it without data loss.
- **Handoff**: The data is now fully prepared for Exploratory Data Analysis, spatial choropleth mapping, and econometric modeling in Week 3.